# Bengaluru Commuting & Road Traffic — Phase 1
**Purpose:** Build an evidence-led baseline of commuting and road-vehicle patterns across Bengaluru, with weekday/weekend comparisons where data supports them.

**Scope**
- Geography: Bengaluru city / urban area, with dataset-specific boundaries documented.
- Population groups: office/work commuters, school students, college students.
- Road modes: two-wheelers, cars/jeeps/vans, taxis/auto-rickshaws, school/institutional buses, other road vehicles where available.
- Public transport: retained in a separate context table; excluded from the primary *private/road-commuter* totals when the dataset supports mode separation.
- Time: all historical periods available from the source, preserving original dates and frequency.

## Critical evidence rule
Do **not** treat vehicle registrations, annual passenger totals, census mode shares, traffic violations, road crashes, modeled traffic data, and observed hourly traffic counts as interchangeable. Each answers a different question. Do not infer Monday-to-Sunday patterns from annual or cross-sectional data. The notebook labels each dataset by evidence type and only runs day-of-week analysis on records with genuine observation dates/times.

**Expected reality:** publicly accessible sources are fragmented. Census commuting data can help describe worker mode choice but is old (2011) and is not a day-by-day count. Public transport ridership and registered-vehicle series are useful context, not direct counts of all commuters on roads. The notebook records these limitations rather than manufacturing missing values.

## 1. Research questions and measurable outputs

1. What is known about work and education commuting in Bengaluru?
2. What vehicle types are represented in available traffic counts, registration data, and traffic surveys?
3. Which weekdays/weekends have higher observed traffic, and what are the mean/median counts by day?
4. How do public-transport usage indicators change over time? (Context only; excluded from the main road-vehicle totals.)
5. Which conclusions are directly observed, census-derived, or estimates?

**Unit distinctions**
- `people/trips`: people or passenger trips; these are not vehicle counts.
- `vehicles`: vehicle registrations, vehicles passing a screenline, or vehicle-hours depending on source.
- `ridership`: boardings/entries, not necessarily unique people.
- `mode share`: proportion of trips or people by mode, with the source's definition retained.

In [ ]:
# Install if needed in your notebook environment:
# %pip install pandas numpy matplotlib seaborn requests openpyxl pyarrow

from pathlib import Path
from datetime import datetime, timezone
import json, re, time, hashlib, warnings
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

ROOT = Path("bengaluru_commuting_phase1")
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
for p in [RAW, PROCESSED, REPORTS]:
    p.mkdir(parents=True, exist_ok=True)

RUN_DATE = pd.Timestamp.now(tz="Asia/Kolkata").strftime("%Y-%m-%d")
print("Run date:", RUN_DATE)
print("Raw data directory:", RAW.resolve())

## 2. Source register and evidence grading

The sources below were identified during initial research. OpenCity is a non-government open-data portal that republishes datasets from public agencies and other sources; validate the underlying publisher, metadata, license, date range, and geographic coverage for each resource before treating it as official.

| Source / dataset | Intended use | Evidence type | Day-of-week analysis? |
|---|---|---|---|
| Census of India 2011, Table B-28 | Worker commute mode and distance to work; mode categories include two-wheelers, cars/vans, auto/taxi, bus, train, walking, cycling | Official census counts; historical snapshot | No |
| OpenCity Bengaluru mobility dataset catalogue | Discover city mobility resources and their downloadable files | Catalogue / metadata | Only if resource has actual observation dates |
| Bengaluru RTO-wise vehicle registration data | Registered fleet trend and vehicle category context | Administrative stock/registration data | No, unless records contain dated observations at daily frequency |
| Total vehicles registered in Bengaluru | Fleet context | Administrative data | Usually no |
| BMRCL station-wise ridership (reported August 2025) | Public-transport context, kept separate | Station entries/ridership; check definitions | Only if actual date/time observations are present |
| BMTC operations/performance | Public-transport context (fleet/utilisation) | Annual or periodic operational indicators | Only if daily observations are available |
| Bengaluru traffic / mobility GitHub projects | Discovery and exploratory validation only | Non-government; source lineage must be assessed | Only when raw timestamps and measurement method are documented |
| Bengaluru Comprehensive Mobility Plan / traffic survey reports | Vehicle classification, occupancy, trip-purpose and modal-share context | Survey/model report; year and study-area boundary matter | Only if survey data has date/time fields |

Official Census metadata:
https://censusindia.gov.in/nada/index.php/catalog/13983

OpenCity Bengaluru mobility catalogue:
https://data.opencity.in/dataset/?groups=bengaluru&tags=Mobility

OpenCity catalogue filtered to CSV resources:
https://data.opencity.in/dataset/?city=&groups=bengaluru&res_format=CSV&tags=Mobility

Non-government Bengaluru traffic-data project (validate source lineage before analytical use):
https://github.com/traffic-kowshik/traffic-data-bengaluru

Example of a community traffic dataset with stated hourly records (not automatically ground-truth; inspect raw file and provenance):
https://github.com/aswinmangatt/Bengaluru-Traffic-Analysis

**Evidence grades**
- `A`: official primary data with documented methodology and clear geography/time.
- `B`: reputable intermediary with an identifiable underlying government/agency source.
- `C`: community/third-party data with some documentation but uncertain measurement lineage.
- `D`: modeled, synthetic, hand-curated, or insufficiently documented; exclude from factual citywide totals.

In [ ]:
source_register = pd.DataFrame([
    {
        "source_id":"CENSUS_B28_2011",
        "dataset":"Census 2011 B-28: workers by distance and mode of travel",
        "publisher":"Office of the Registrar General & Census Commissioner, India",
        "url":"https://censusindia.gov.in/nada/index.php/catalog/13983",
        "evidence_type":"official census count / cross-sectional",
        "grade":"A",
        "use":"historical worker commute mode context",
        "public_transport_separate":True,
        "supports_day_of_week":False,
        "known_limitations":"2011 snapshot; workers, not school/college students; not daily traffic counts"
    },
    {
        "source_id":"OPENCITY_MOBILITY",
        "dataset":"OpenCity Bengaluru mobility catalogue",
        "publisher":"OpenCity; underlying publisher varies by resource",
        "url":"https://data.opencity.in/dataset/?groups=bengaluru&tags=Mobility",
        "evidence_type":"catalogue; individual resource varies",
        "grade":"B pending resource-level verification",
        "use":"discover downloadable traffic, vehicle registration, mobility and transit datasets",
        "public_transport_separate":True,
        "supports_day_of_week":"resource-dependent",
        "known_limitations":"verify original publisher, definitions, coverage, licence, date range"
    },
    {
        "source_id":"GITHUB_TRAFFIC_KOWSHIK",
        "dataset":"Traffic Data Bengaluru repository",
        "publisher":"Community GitHub project",
        "url":"https://github.com/traffic-kowshik/traffic-data-bengaluru",
        "evidence_type":"third-party repository / resource-dependent",
        "grade":"C pending provenance",
        "use":"route/transport data discovery; exploratory only",
        "public_transport_separate":True,
        "supports_day_of_week":"only if raw observations have real timestamps",
        "known_limitations":"repository is not itself proof of measured traffic counts"
    },
    {
        "source_id":"GITHUB_TRAFFIC_ANALYSIS",
        "dataset":"Bengaluru Traffic Analysis example project",
        "publisher":"Community GitHub project",
        "url":"https://github.com/aswinmangatt/Bengaluru-Traffic-Analysis",
        "evidence_type":"third-party traffic dataset / source lineage to validate",
        "grade":"C/D until independently validated",
        "use":"candidate dataset for schema/provenance review, not default citywide truth",
        "public_transport_separate":True,
        "supports_day_of_week":"claimed hourly schema; validate raw data and collection method",
        "known_limitations":"repository descriptions alone do not establish that records are real measured counts"
    }
])
source_register

## 3. Discover OpenCity datasets through its CKAN API

This cell queries the public CKAN catalogue and saves dataset metadata. It does not assume that every listed resource is a direct government source or that a downloadable CSV is suitable for analysis.

If the API is temporarily unavailable, use the catalogue URLs in the source register and save resources manually into `data/raw/`. Keep original filenames and record the download URL.

In [ ]:
CKAN_API = "https://data.opencity.in/api/3/action/package_search"
queries = [
    "Bengaluru mobility",
    "Bengaluru vehicle registrations",
    "Bengaluru traffic",
    "BMRCL station-wise ridership",
    "BMTC operation performance",
    "Bengaluru mobility indicators",
    "Bengaluru RTO-wise vehicle registration",
    "Total vehicles registered Bengaluru",
    "Bengaluru traffic violations",
    "Bengaluru road crashes",
]

catalogue_rows = []
session = requests.Session()
session.headers.update({"User-Agent":"BengaluruCommutingResearch/1.0 (research notebook)"})

for q in queries:
    try:
        resp = session.get(CKAN_API, params={"q": q, "rows": 100}, timeout=30)
        resp.raise_for_status()
        payload = resp.json()
        if not payload.get("success"):
            print("Catalogue query failed:", q)
            continue
        for pkg in payload["result"].get("results", []):
            for resource in pkg.get("resources", []):
                catalogue_rows.append({
                    "query": q,
                    "package_name": pkg.get("name"),
                    "package_title": pkg.get("title"),
                    "organization": (pkg.get("organization") or {}).get("title"),
                    "package_url": "https://data.opencity.in/dataset/" + str(pkg.get("name","")),
                    "resource_name": resource.get("name"),
                    "format": resource.get("format"),
                    "resource_url": resource.get("url"),
                    "resource_id": resource.get("id"),
                    "created": resource.get("created"),
                    "last_modified": resource.get("last_modified"),
                    "description": resource.get("description"),
                })
    except Exception as e:
        print(f"Could not query {q!r}: {type(e).__name__}: {e}")

catalogue = pd.DataFrame(catalogue_rows).drop_duplicates(
    subset=["package_name", "resource_id"]
) if catalogue_rows else pd.DataFrame()

catalogue.to_csv(PROCESSED / "opencity_catalogue_discovery.csv", index=False)
print("Resources discovered:", len(catalogue))
display(catalogue[["package_title","organization","resource_name","format","resource_url","last_modified"]].head(50)
        if not catalogue.empty else "No API results. Use the catalogue links manually.")

## 4. Select and download candidate resources

Download only common tabular formats and preserve the raw file. A successful HTTP response does not guarantee the file is clean or trustworthy; each downloaded file must be inspected. The default below is intentionally conservative and does not recursively download every resource in the catalogue.

Edit `RESOURCE_ALLOWLIST` after reviewing the catalogue. Put a resource ID in the list only after confirming its title, publisher, licence and intended use.

In [ ]:
# Start empty to avoid downloading unreviewed resources.
# Example after reviewing catalogue:
# RESOURCE_ALLOWLIST = ["resource-id-1", "resource-id-2"]
RESOURCE_ALLOWLIST = []

def safe_filename(value, max_len=100):
    value = re.sub(r"[^A-Za-z0-9._-]+", "_", str(value)).strip("._")
    return (value or "resource")[:max_len]

download_log = []

if not catalogue.empty and RESOURCE_ALLOWLIST:
    chosen = catalogue[catalogue["resource_id"].isin(RESOURCE_ALLOWLIST)]
    for _, row in chosen.iterrows():
        url = row.get("resource_url")
        if not isinstance(url, str) or not url.startswith(("https://","http://")):
            continue
        ext = str(row.get("format","")).lower().strip()
        if ext not in {"csv","xlsx","xls","json","geojson","zip","pdf"}:
            print("Skipped unsupported/unknown format:", row.get("resource_name"), ext)
            continue
        fname = safe_filename(row.get("package_title")) + "__" + safe_filename(row.get("resource_name"))
        suffix = "." + ext
        path = RAW / (fname if fname.lower().endswith(suffix) else fname + suffix)
        try:
            r = session.get(url, timeout=90)
            r.raise_for_status()
            path.write_bytes(r.content)
            download_log.append({
                "resource_id":row.get("resource_id"), "url":url,
                "path":str(path), "bytes":len(r.content),
                "sha256":hashlib.sha256(r.content).hexdigest(),
                "downloaded_at_utc":datetime.now(timezone.utc).isoformat(),
                "http_status":r.status_code
            })
            print("Saved", path, f"({len(r.content):,} bytes)")
        except Exception as e:
            print("Download failed:", url, type(e).__name__, e)
else:
    print("No resources downloaded automatically. Review catalogue and populate RESOURCE_ALLOWLIST.")

pd.DataFrame(download_log).to_csv(PROCESSED / "download_log.csv", index=False)

## 5. Census commuting data — metadata and safe handling

Census Table B-28 records workers by mode of travel and distance to work. Its modes include walking, bicycle, moped/scooter/motorcycle, car/jeep/van, auto-rickshaw/taxi, bus and train. It is useful for historic worker mode shares, but does not cover school/college students as a separate population and has no weekday/date-of-travel observations.

The Census download page exposes an XLSX resource. Because download endpoints can change, use the official catalogue page and download the Excel file manually if the next cell cannot locate a working resource. Do not silently substitute a third-party file.

In [ ]:
CENSUS_PAGE = "https://censusindia.gov.in/nada/index.php/catalog/13983"
CENSUS_FILE = RAW / "census_2011_B28_karnataka_workers_mode_distance.xlsx"

# First try to locate a spreadsheet link on the official metadata page.
try:
    from bs4 import BeautifulSoup
    page = session.get(CENSUS_PAGE, timeout=30)
    page.raise_for_status()
    soup = BeautifulSoup(page.text, "html.parser")
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        label = (a.get_text(" ", strip=True) + " " + href).lower()
        if any(x in label for x in [".xlsx", "ddw-2900b-28", "download"]):
            full = requests.compat.urljoin(CENSUS_PAGE, href)
            links.append((a.get_text(" ", strip=True), full))
    print("Candidate official Census download links:")
    for item in links[:20]:
        print(item)
except Exception as e:
    print("Could not auto-discover Census download link:", type(e).__name__, e)

if CENSUS_FILE.exists():
    print("Census workbook already present:", CENSUS_FILE)
else:
    print("If needed, download the B-28 workbook from the official page above and save it as:")
    print(CENSUS_FILE.resolve())

In [ ]:
# Inspect the Census workbook if you downloaded it.
if CENSUS_FILE.exists():
    xls = pd.ExcelFile(CENSUS_FILE)
    print("Sheets:", xls.sheet_names)
    for sheet in xls.sheet_names:
        preview = pd.read_excel(CENSUS_FILE, sheet_name=sheet, header=None, nrows=12)
        print("\\nSHEET:", sheet)
        display(preview)
else:
    print("Skipped: official Census workbook not present locally.")

## 6. Load downloaded tabular files and profile schemas

This generic loader supports CSV, Excel, JSON and Parquet. PDF files are listed but not automatically converted to data because table extraction requires manual quality review. The code records row counts and column names; it does not guess the meaning of columns.

In [ ]:
def load_tabular(path):
    suffix = path.suffix.lower()
    if suffix == ".csv":
        # Try common encodings; preserve the first successful parse.
        last_error = None
        for enc in ["utf-8-sig", "utf-8", "cp1252", "latin1"]:
            try:
                return pd.read_csv(path, encoding=enc, low_memory=False)
            except Exception as e:
                last_error = e
        raise last_error
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix == ".json":
        return pd.read_json(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported tabular format: {suffix}")

raw_tables = {}
schema_rows = []
for path in sorted(RAW.glob("*")):
    if path.suffix.lower() not in {".csv",".xlsx",".xls",".json",".parquet"}:
        continue
    try:
        df = load_tabular(path)
        raw_tables[path.name] = df
        schema_rows.append({
            "file":path.name, "rows":len(df), "columns":len(df.columns),
            "columns_list":" | ".join(map(str, df.columns)),
            "missing_cells":int(df.isna().sum().sum()),
            "duplicate_rows":int(df.duplicated().sum())
        })
        print(f"{path.name}: {df.shape}")
    except Exception as e:
        print("Could not parse", path.name, type(e).__name__, e)

schema_profile = pd.DataFrame(schema_rows)
display(schema_profile)
schema_profile.to_csv(PROCESSED / "schema_profile.csv", index=False)

## 7. Evidence classification before analysis

Create a manual mapping for each file. This step is important: a vehicle-registration stock series must not be treated as vehicles travelling on a particular Tuesday. Similarly, BMTC/BMRCL ridership is a separate public-transport indicator and must not be added to private road vehicles.

Use these recommended `measure_type` values:
- `observed_traffic_count`: observed vehicles passing a location during a stated interval.
- `observed_person_count`: observed people or passenger trips during a stated interval.
- `survey_estimate`: estimate expanded from a survey or origin-destination model.
- `mode_share`: proportions by travel mode, not raw counts.
- `registered_vehicle_stock`: vehicles registered as of a period.
- `public_transport_ridership`: passenger boardings/entries.
- `operations_metric`: fleet size, service km, utilisation, etc.
- `proxy_or_modelled`: modeled / inferred values.
- `unknown`: not yet validated; exclude from quantitative conclusions.

In [ ]:
# Fill this after inspecting schema_profile and source metadata.
# Keep "unknown" until the meaning and provenance are confirmed.
file_metadata = pd.DataFrame([
    # Example structure (not data):
    # {"file":"my_traffic.csv", "source_url":"...", "publisher":"...",
    #  "measure_type":"observed_traffic_count", "evidence_grade":"B",
    #  "geography":"...", "period_start":"YYYY-MM-DD", "period_end":"YYYY-MM-DD",
    #  "time_column":"timestamp", "value_column":"vehicle_count",
    #  "mode_column":"vehicle_type", "public_transport":False,
    #  "methodology_notes":"..."}
], columns=[
    "file","source_url","publisher","measure_type","evidence_grade","geography",
    "period_start","period_end","time_column","value_column","mode_column",
    "public_transport","methodology_notes"
])
file_metadata.to_csv(PROCESSED / "file_metadata.csv", index=False)
display(file_metadata)

## 8. Standardize actual timestamped traffic observations

The next function only prepares a dataset after you explicitly identify its timestamp and count columns. It does not manufacture dates, interpolate missing days, or convert registrations into traffic counts.

A row can represent a location/hour, a vehicle class/hour, or a daily total. Document that granularity before grouping.

In [ ]:
def standardize_traffic(df, timestamp_col, count_col, location_col=None,
                        vehicle_type_col=None, public_transport_col=None):
    required = [timestamp_col, count_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns: {missing}")

    out = pd.DataFrame()
    out["timestamp"] = pd.to_datetime(df[timestamp_col], errors="coerce", dayfirst=False)
    out["vehicle_count"] = pd.to_numeric(
        df[count_col].astype(str).str.replace(",", "", regex=False), errors="coerce"
    )
    out["location"] = df[location_col].astype(str) if location_col and location_col in df else "unspecified"
    out["vehicle_type"] = df[vehicle_type_col].astype(str) if vehicle_type_col and vehicle_type_col in df else "unspecified"
    if public_transport_col and public_transport_col in df:
        out["public_transport"] = df[public_transport_col].astype(str).str.lower().isin(
            ["true","1","yes","public transport","bus","metro","train"]
        )
    else:
        out["public_transport"] = False

    out = out.dropna(subset=["timestamp","vehicle_count"])
    out = out[out["vehicle_count"] >= 0].copy()
    out["date"] = out["timestamp"].dt.date
    out["day_of_week"] = out["timestamp"].dt.day_name()
    out["weekday_type"] = np.where(out["timestamp"].dt.dayofweek < 5, "Weekday",
                                   np.where(out["timestamp"].dt.dayofweek == 5, "Saturday", "Sunday"))
    out["hour"] = out["timestamp"].dt.hour
    return out

# Example once a validated file has been selected:
# raw_df = raw_tables["validated_traffic_file.csv"]
# traffic = standardize_traffic(raw_df, timestamp_col="observed_at",
#                               count_col="vehicle_count",
#                               location_col="junction",
#                               vehicle_type_col="vehicle_class")

## 9. Weekday / weekend analysis for genuine traffic counts

This analysis runs only on a standardized dataset with true observation timestamps. If data contains multiple locations, the default is to report both (a) per-location averages and (b) sums only when locations are non-overlapping screenlines or a defensible citywide counting design. Summing overlapping sensors or multiple observations of the same vehicles can double-count traffic.

The daily mean is calculated over observed days only. The number of observed days is shown alongside the mean so incomplete coverage is visible.

In [ ]:
def summarize_day_patterns(traffic, count_is_interval_volume=True):
    required = {"timestamp","vehicle_count","location","vehicle_type","public_transport","date","day_of_week","weekday_type"}
    missing = required - set(traffic.columns)
    if missing:
        raise ValueError(f"Traffic data is not standardized; missing: {missing}")

    # Main analysis excludes public transport records if they have been explicitly classified.
    road = traffic.loc[~traffic["public_transport"]].copy()
    public = traffic.loc[traffic["public_transport"]].copy()

    daily = (road.groupby(["date","location"], as_index=False)
             .agg(road_vehicle_count=("vehicle_count","sum"),
                  records=("vehicle_count","size")))
    daily["date"] = pd.to_datetime(daily["date"])
    daily["day_of_week"] = daily["date"].dt.day_name()
    daily["weekday_type"] = np.where(daily["date"].dt.dayofweek < 5, "Weekday",
                                     np.where(daily["date"].dt.dayofweek == 5, "Saturday", "Sunday"))

    # Per-location summaries; safer than citywide summing when sensor coverage is unknown.
    day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
    by_day = (daily.groupby("day_of_week")
              .agg(mean_daily_count=("road_vehicle_count","mean"),
                   median_daily_count=("road_vehicle_count","median"),
                   observed_location_days=("road_vehicle_count","count"))
              .reindex(day_order).reset_index())
    by_daytype = (daily.groupby("weekday_type")
                  .agg(mean_location_day_count=("road_vehicle_count","mean"),
                       median_location_day_count=("road_vehicle_count","median"),
                       observed_location_days=("road_vehicle_count","count"))
                  .reset_index())
    return road, public, daily, by_day, by_daytype

# Example:
# road, public, daily_counts, weekday_summary, weekend_summary = summarize_day_patterns(traffic)
# display(weekday_summary, weekend_summary)

In [ ]:
def plot_day_patterns(weekday_summary, weekend_summary):
    if not weekday_summary.empty:
        plt.figure(figsize=(10,4))
        plt.bar(weekday_summary["day_of_week"], weekday_summary["mean_daily_count"])
        plt.xticks(rotation=35, ha="right")
        plt.ylabel("Mean count per observed location-day")
        plt.title("Observed road-vehicle counts by day of week")
        plt.tight_layout()
        plt.show()

    if not weekend_summary.empty:
        plt.figure(figsize=(7,4))
        plt.bar(weekend_summary["weekday_type"], weekend_summary["mean_location_day_count"])
        plt.ylabel("Mean count per observed location-day")
        plt.title("Observed road-vehicle counts: weekday vs Saturday vs Sunday")
        plt.tight_layout()
        plt.show()

# plot_day_patterns(weekday_summary, weekend_summary)

## 10. Vehicle-type analysis

Do not assume a source's `vehicle_type` field means the same thing across datasets. Map raw labels into common groups only after checking the source's codebook. Keep buses as a separate class and decide whether the question is *all road vehicles* or *non-public-transport road vehicles*. For this project's primary total, exclude public transport passengers from passenger totals and report public-transport vehicles/operations separately rather than silently discarding buses from a road vehicle count.

Recommended categories:
- `two_wheeler`
- `car_jeep_van`
- `auto_taxi`
- `school_or_institutional_bus`
- `public_bus`
- `goods_vehicle`
- `other_or_unknown`

A public bus is physically on the road, so it may be included in an **all-road-vehicles** count but must be separated from private/non-public-transport vehicles. Do not add passenger ridership to vehicle counts.

In [ ]:
VEHICLE_LABEL_MAP = {
    # Add normalized mappings only after reviewing original labels.
    "motorcycle":"two_wheeler",
    "motorbike":"two_wheeler",
    "scooter":"two_wheeler",
    "moped":"two_wheeler",
    "car":"car_jeep_van",
    "jeep":"car_jeep_van",
    "van":"car_jeep_van",
    "auto rickshaw":"auto_taxi",
    "autorickshaw":"auto_taxi",
    "taxi":"auto_taxi",
    "school bus":"school_or_institutional_bus",
    "institutional bus":"school_or_institutional_bus",
    "city bus":"public_bus",
    "public bus":"public_bus",
    "truck":"goods_vehicle",
    "lorry":"goods_vehicle",
}

def normalize_vehicle_type(value):
    label = str(value).strip().lower()
    return VEHICLE_LABEL_MAP.get(label, "other_or_unknown")

# Example:
# traffic["vehicle_group"] = traffic["vehicle_type"].map(normalize_vehicle_type)
# vehicle_mix = (traffic[~traffic["public_transport"]]
#                .groupby("vehicle_group")["vehicle_count"].sum()
#                .sort_values(ascending=False).rename("count").reset_index())
# display(vehicle_mix)

## 11. Public transport context — separate from the main road-commuter total

Maintain a separate table for BMTC, BMRCL, suburban rail and other public transport. Depending on the source, the measure may be passenger boardings, entries, fleet size, trips operated, route-km, or service frequency. Never add boardings to road vehicle counts. If public buses are present in traffic count data, keep them in the all-road-vehicles view and optionally exclude them for a private/non-public-transport view.

Suggested context fields:
`date_or_period`, `agency`, `measure`, `value`, `unit`, `geography`, `source_url`, `source_grade`, `notes`.

In [ ]:
public_transport_context = pd.DataFrame(columns=[
    "date_or_period","agency","measure","value","unit","geography",
    "source_url","source_grade","notes"
])
public_transport_context.to_csv(PROCESSED / "public_transport_context.csv", index=False)
display(public_transport_context)

## 12. Worker and student commuter estimates — prevent false precision

Census worker mode shares are not enough to estimate current daily student trips or weekday counts. For office, school and college commuting, use separate population denominators and clearly named assumptions.

A defensible estimate needs, at minimum:
1. population/enrolment or employment count and year;
2. share that travels on a given day;
3. share of each transport mode;
4. average occupancy for vehicle conversion;
5. school/college/office calendar, holidays and remote-work effects;
6. a defined city boundary and whether commuters entering from outside the city are counted.

If any of these are missing, publish a range or leave the estimate blank. Do not present a modeled estimate as an observed traffic count.

In [ ]:
# Empty schema for future estimates. No values are fabricated in this notebook.
commuter_estimates = pd.DataFrame(columns=[
    "population_group", # office_workers / school_students / college_students
    "reference_year",
    "geography",
    "population_or_enrolment",
    "share_travelling_on_day",
    "mode",
    "mode_share",
    "average_occupancy",
    "estimated_people_trips_per_day",
    "estimated_vehicles_per_day",
    "estimate_low",
    "estimate_high",
    "source_url",
    "assumptions",
    "evidence_type" # should be survey_estimate, not observed count
])
commuter_estimates.to_csv(PROCESSED / "commuter_estimates_template.csv", index=False)

## 13. Quality checks and audit trail

Before any chart or claim is published:
- confirm raw file publisher, source URL, retrieval date, licence, data period and geographic boundary;
- inspect the data dictionary and units;
- check duplicates, missing timestamps, impossible negative values, repeated time/location keys and timezone;
- distinguish zero traffic from missing data;
- check whether counts are per 5/15/60 minutes, daily, or cumulative;
- check whether different sensors cover overlapping traffic;
- check holiday, school calendar, examination, strike, weather, and special-event effects where possible;
- document any manual mapping and retain the original raw labels;
- include `n observed days` / `n locations` with every daily mean.

In [ ]:
def quality_report(df, timestamp_col=None, count_col=None, key_cols=None):
    report = {
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum()),
    }
    if timestamp_col and timestamp_col in df:
        ts = pd.to_datetime(df[timestamp_col], errors="coerce")
        report["invalid_timestamps"] = int(ts.isna().sum())
        report["min_timestamp"] = str(ts.min())
        report["max_timestamp"] = str(ts.max())
    if count_col and count_col in df:
        values = pd.to_numeric(df[count_col], errors="coerce")
        report["invalid_numeric_counts"] = int(values.isna().sum())
        report["negative_counts"] = int((values < 0).sum())
    if key_cols and all(c in df for c in key_cols):
        report["duplicate_keys"] = int(df.duplicated(subset=key_cols).sum())
    return pd.Series(report, name="quality_report")

# Example after selecting a source:
# quality_report(raw_df, timestamp_col="timestamp", count_col="count",
#                key_cols=["timestamp","location","vehicle_type"])

## 14. Export analysis outputs

Use CSV outputs for reproducibility and sharing. Keep raw files untouched; transformed files should have clear names and the run date. Only export day-of-week summaries after a real, validated timestamped traffic dataset has been loaded.

In [ ]:
# Save the source register and catalogue discovery output even if no traffic file is yet usable.
source_register.to_csv(PROCESSED / "source_register.csv", index=False)

# Example after validated analysis:
# road.to_csv(PROCESSED / f"road_traffic_observations_{RUN_DATE}.csv", index=False)
# daily_counts.to_csv(REPORTS / f"daily_road_vehicle_counts_{RUN_DATE}.csv", index=False)
# weekday_summary.to_csv(REPORTS / f"weekday_road_vehicle_summary_{RUN_DATE}.csv", index=False)
# weekend_summary.to_csv(REPORTS / f"weekday_weekend_comparison_{RUN_DATE}.csv", index=False)

print("Notebook workspace initialized:", ROOT.resolve())
print("No synthetic commuter/traffic observations have been created.")

## Google Maps Platform: historical-data limitation and collection plan

Google Routes API does not provide a downloadable archive of arbitrary past driving traffic by date. Its `departureTime` for driving must be current or future. `staticDuration` is a route-duration baseline, not an observation for a particular Monday, Sunday, season, or holiday week.

Keep these data types separate:
1. **Google route baseline**: route-duration baseline retrieved now; not day-specific historical traffic.
2. **Prospective Google observations**: API snapshots collected at fixed times going forward.
3. **Actual historical traffic**: must come from a licensed source that has dated records.

Deepavali 2026 is listed for Sunday, 8 November in the Bengaluru holiday calendar. Proposed windows:
- Pre-Deepavali: 1–7 November 2026.
- Post-Deepavali: 9–15 November 2026.
- All-season comparison: separately select one seven-day summer week, one monsoon week and one winter week. A single seven-day window cannot represent all seasons.

These windows are a sampling plan, not historical Google query dates. Do not fabricate observations or copy today's baseline into past dates.

### Google Routes API setup

Enable Routes API and billing in Google Cloud, then set `GOOGLE_MAPS_API_KEY` as an environment variable. Do not hardcode the key in the notebook. Google Maps Platform content is subject to applicable terms and storage restrictions; review the terms for your account before saving or redistributing responses.

In [ ]:
import os, time
from datetime import datetime
from zoneinfo import ZoneInfo
from pathlib import Path

import pandas as pd
import requests

GOOGLE_ROUTES_ENDPOINT = "https://routes.googleapis.com/directions/v2:computeRoutes"
GOOGLE_MAPS_API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")
GOOGLE_DATA_DIR = Path("bengaluru_commuting_phase1") / "data" / "google_routes"
GOOGLE_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Approximate sample corridors; review exact route endpoints before use.
route_catalogue = pd.DataFrame([
    {"route_id":"ORR_MARATHAHALLI_BELLANDUR","origin":"Marathahalli, Bengaluru, Karnataka","destination":"Bellandur, Bengaluru, Karnataka","purpose":"ORR / IT corridor"},
    {"route_id":"WHITEFIELD_MARATHAHALLI","origin":"ITPL, Whitefield, Bengaluru, Karnataka","destination":"Marathahalli Bridge, Bengaluru, Karnataka","purpose":"East Bengaluru commute"},
    {"route_id":"ELECTRONIC_CITY_SILK_BOARD","origin":"Electronic City Phase 1, Bengaluru, Karnataka","destination":"Silk Board Junction, Bengaluru, Karnataka","purpose":"South Bengaluru commute"},
    {"route_id":"SARJAPUR_BELLANDUR","origin":"Sarjapur, Bengaluru, Karnataka","destination":"Bellandur, Bengaluru, Karnataka","purpose":"South-east corridor"},
    {"route_id":"HEBBAL_MG_ROAD","origin":"Hebbal Flyover, Bengaluru, Karnataka","destination":"MG Road Metro Station, Bengaluru, Karnataka","purpose":"North-to-central corridor"},
    {"route_id":"YELAHANKA_HEBBAL","origin":"Yelahanka, Bengaluru, Karnataka","destination":"Hebbal Flyover, Bengaluru, Karnataka","purpose":"North Bengaluru commute"},
    {"route_id":"KENGERI_MYSORE_ROAD","origin":"Kengeri, Bengaluru, Karnataka","destination":"Mysore Road Metro Station, Bengaluru, Karnataka","purpose":"West Bengaluru commute"},
    {"route_id":"HOSKOTE_WHITEFIELD","origin":"Hoskote, Karnataka","destination":"Whitefield, Bengaluru, Karnataka","purpose":"Eastern suburb corridor"},
])
display(route_catalogue)
print("API key configured:", bool(GOOGLE_MAPS_API_KEY))

In [ ]:
def parse_duration_seconds(value):
    if not value:
        return None
    value = str(value).strip()
    return float(value[:-1]) if value.endswith("s") else None

def google_route_baseline(origin, destination):
    '''
    Retrieve a route baseline. This is NOT a dated historical traffic record,
    and it does NOT return vehicle counts.
    '''
    if not GOOGLE_MAPS_API_KEY:
        raise RuntimeError("Set GOOGLE_MAPS_API_KEY in your environment first.")

    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": GOOGLE_MAPS_API_KEY,
        "X-Goog-FieldMask": "routes.distanceMeters,routes.duration,routes.staticDuration"
    }
    body = {
        "origin": {"address": origin},
        "destination": {"address": destination},
        "travelMode": "DRIVE",
        "routingPreference": "TRAFFIC_UNAWARE"
    }
    response = requests.post(GOOGLE_ROUTES_ENDPOINT, headers=headers, json=body, timeout=30)
    response.raise_for_status()
    routes = response.json().get("routes", [])
    if not routes:
        return {"status":"no_route", "origin":origin, "destination":destination}

    route = routes[0]
    return {
        "status":"ok",
        "origin":origin,
        "destination":destination,
        "distance_meters":route.get("distanceMeters"),
        "duration_seconds":parse_duration_seconds(route.get("duration")),
        "static_duration_seconds":parse_duration_seconds(route.get("staticDuration")),
        "evidence_type":"google_route_baseline_snapshot",
        "historical_date_specific":False,
        "vehicle_count_available":False
    }

# API calls are disabled by default to avoid unexpected billing.
RUN_GOOGLE_BASELINE = False

if RUN_GOOGLE_BASELINE:
    rows = []
    for _, r in route_catalogue.iterrows():
        record = google_route_baseline(r["origin"], r["destination"])
        record.update({
            "route_id":r["route_id"],
            "purpose":r["purpose"],
            "collected_at_local":datetime.now(ZoneInfo("Asia/Kolkata")).isoformat()
        })
        rows.append(record)
        time.sleep(0.15)
    google_baseline_df = pd.DataFrame(rows)
    google_baseline_df.to_csv(GOOGLE_DATA_DIR / "google_route_baseline_snapshot.csv", index=False)
    display(google_baseline_df)
else:
    print("No API calls made. Review route list and billing, then set RUN_GOOGLE_BASELINE=True.")

In [ ]:
# Sampling calendar only; no historical traffic observations are created here.
sampling_windows = pd.DataFrame([
    {"window_id":"SUMMER_BASELINE","start_date":None,"end_date":None,"purpose":"Choose representative 7-day summer week","status":"date_to_be_selected"},
    {"window_id":"MONSOON_BASELINE","start_date":None,"end_date":None,"purpose":"Choose representative 7-day monsoon week","status":"date_to_be_selected"},
    {"window_id":"WINTER_BASELINE","start_date":None,"end_date":None,"purpose":"Choose representative 7-day winter week","status":"date_to_be_selected"},
    {"window_id":"PRE_DEEPAVALI_2026","start_date":"2026-11-01","end_date":"2026-11-07","purpose":"Seven days before Deepavali (listed 2026-11-08)","status":"future collection required"},
    {"window_id":"POST_DEEPAVALI_2026","start_date":"2026-11-09","end_date":"2026-11-15","purpose":"Seven days after Deepavali","status":"future collection required"},
])
display(sampling_windows)

### Interpretation rules

- Never create synthetic rows for historical dates.
- Never copy today's route baseline into a past date.
- Route duration is not vehicle count or commuter count.
- To compare actual weekday/weekend conditions, collect prospective snapshots at fixed times or obtain a properly licensed third-party dataset containing dated observations.
- The primary road-vehicle count analysis still requires actual counts from traffic sensors, government surveys, or suitable open datasets.

## 15. Phase 1 deliverable checklist

- [x] Research questions, definitions and scope established.
- [x] Initial source register created with evidence grades and limitations.
- [x] OpenCity catalogue discovery and download workflow included.
- [x] Census commuting source documented; official workbook can be inspected.
- [x] Public transport context separated from the primary road-commuter analysis.
- [x] Day-of-week/weekend analysis implemented for validated, timestamped traffic counts.
- [x] Data quality checks and export templates included.
- [ ] Populate allowlist and download the specific source files after metadata review.
- [ ] Confirm which sources contain observed dated vehicle counts, rather than registrations or modeled values.
- [ ] Obtain/validate daily traffic counts across enough locations and dates for citywide comparisons.
- [ ] Obtain separate school/college enrolment and employment denominators before producing commuter estimates.

**Interpretation boundary:** This is a data-acquisition and analysis framework, not a claim that a complete citywide daily count dataset already exists publicly. Until validated dated observations are loaded, the weekday/weekend charts remain intentionally unexecuted.